<a href="https://colab.research.google.com/github/KingExecutioner/Projects/blob/main/Matric_Exam_Paper_Scraper_for_Google_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Matric Exam Paper Scraper for Google Colab

This script is designed to scrape PDF exam papers from South African educational portals, organize them by subject and year, and save them directly to your Google Drive.

### Instructions:

1.  **Run the first two code cells** below to install dependencies and mount your Google Drive.
2.  **Edit the 'Configuration (User Inputs)' section** in the main scraping script cell (Cell 3):
    *   **`BASE_URL`**: Replace the placeholder URL with the actual URL of the page you want to scrape (e.g., a page listing exam papers for a specific year or subject).
    *   **`TARGET_SUBJECTS`**: Modify the list to include the specific subjects you are interested in (e.g., `["Mathematics", "Physical Sciences"]`). Use `["All"]` to attempt to download all subjects found.
    *   **`TARGET_YEARS`**: Adjust the list of years you want to target (e.g., `[2021, 2022, 2023]`).
3.  **Run the main scraping script cell** (Cell 3). The script will print progress and download files to your Google Drive.

### Important Notes:
*   **Website Structure**: The script uses heuristics to identify document types (Question Paper, Memorandum, Addendum), subjects, and years from link text and filenames. These heuristics might need slight adjustments if the target website's naming conventions are different.
*   **Rate Limiting**: A `time.sleep(1)` delay is included between requests to be polite to the server. Do not remove or significantly reduce this delay.
*   **User-Agent**: A custom `User-Agent` is set to mimic a browser, which helps prevent denial from some servers.
*   **Error Handling**: Basic `try-except` blocks are in place for network requests and file operations.
*   **Google Drive Permissions**: Ensure you grant Google Colab permission to access your Google Drive when prompted.

In [1]:
# Install necessary libraries
!pip install requests beautifulsoup4

In [2]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Main Exam Paper Scraping Script

This cell contains the core logic for fetching HTML, parsing links, filtering PDFs, extracting metadata, and downloading files to your Google Drive. Remember to configure the `BASE_URL`, `TARGET_SUBJECTS`, and `TARGET_YEARS` below before running.

In [3]:
# --- Imports ---
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
import os
import time
import re
import random # Added for randomized delay

# --- Configuration (User Inputs) ---
# IMPORTANT: Update these variables for your specific scraping needs.
# Replace with the URL of the page you want to scrape. This should be a page
# that lists or links to the exam papers.
BASE_URL = "https://www.saexampapers.co.za/grade-12-english/"

# List of subjects to filter. Use ["All"] to attempt to download all found.
# Example: ["Mathematics", "Physical Sciences", "English Home Language"]
TARGET_SUBJECTS = ["All"]

# List of years to filter. The script will try to extract years from link text/filenames.
# Example: [2021, 2022, 2023]
TARGET_YEARS = []

# Google Drive base path for saving files. Files will be saved under this path in structured folders.
DRIVE_BASE_PATH = "/content/drive/MyDrive/Matric_Past_Papers"

# Custom User-Agent to be polite and avoid server denial.
# Using a more generic browser User-Agent and additional headers to reduce chances of being blocked.
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36', # Updated User-Agent to a very recent version
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7',
    'Accept-Language': 'en-US,en;q=0.9',
    'Accept-Encoding': 'gzip, deflate, br',
    'Connection': 'keep-alive',
    'Upgrade-Insecure-Requests': '1',
    'Referer': BASE_URL, # Add Referer header to appear as if coming from the same site
    'DNT': '1', # Do Not Track request header
    'Sec-Fetch-Dest': 'document',
    'Sec-Fetch-Mode': 'navigate',
    'Sec-Fetch-Site': 'same-origin',
    'Sec-Fetch-User': '?1',
    'sec-ch-ua': '"Chromium";v="124", "Google Chrome";v="124", "Not-A.Brand";v="99"', # Updated client hints
    'sec-ch-ua-mobile': '?0',
    'sec-ch-ua-platform': '"Windows"'
}

# Rate limiting: delay between requests (in seconds) to prevent overwhelming the server.
# Increased base delay and introduced randomness
REQUEST_DELAY_SECONDS_MIN = 2
REQUEST_DELAY_SECONDS_MAX = 5

# Create a session object to maintain headers and cookies across requests
session = requests.Session()
session.headers.update(HEADERS) # Set session headers initially

# --- Helper Functions ---

def get_page_content(url):
    """Fetches the content of a given URL with error handling and rate limiting."""
    try:
        print(f"Fetching URL: {url}")
        # Use the session object for requests
        response = session.get(url, timeout=10)
        response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)
        # Introduce random delay
        time.sleep(random.uniform(REQUEST_DELAY_SECONDS_MIN, REQUEST_DELAY_SECONDS_MAX))
        return response.text
    except requests.exceptions.RequestException as e:
        print(f"ERROR fetching {url}: {e}")
        return None

def identify_doc_type(link_text, filename):
    """
    Identifies the document type (Question Paper, Memorandum, Addendum, or Other)
    based on link text and filename using keywords.
    """
    lower_text = (link_text + " " + filename).lower()

    if any(keyword in lower_text for keyword in ['memo', 'memorandum', 'solution', 'marking guideline', 'marking-guideline']):
        return "Memorandum"
    elif any(keyword in lower_text for keyword in ['question', 'paper', 'p1', 'p2', 'paper1', 'paper2', 'qp']):
        return "Question_Paper"
    elif 'addendum' in lower_text:
        return "Addendum"
    return "Other_Document_Type"

def extract_year_from_text(text):
    """
    Extracts a 4-digit year from the given text.
    """
    years = re.findall(r'\b(20\d{2}|19\d{2})\b', text) # Finds years like 19XX or 20XX
    # Prioritize years within the typical range for exam papers if multiple are found
    for year in sorted(map(int, years), reverse=True):
        if 1990 <= year <= 2050: # Reasonable range for past papers
            return year
    return None

def extract_subject_from_text(text):
    """
    Tries to extract a subject name from the text based on common South African Matric subjects.
    This is a heuristic and might need to be refined based on the target website's structure.
    """
    # Common South African Matric subjects (can be extended)
    subjects = [
        "Mathematics", "Physical Sciences", "Life Sciences", "English Home Language",
        "Afrikaans Home Language", "IsiXhosa Home Language", "IsiZulu Home Language",
        "Setswana Home Language", "Sepedi Home Language", "Sesotho Home Language",
        "Xitsonga Home Language", "Tshivenda Home Language", "SiSwati Home Language",
        "Ndebele Home Language", "Accounting", "Business Studies", "Economics", "Geography", "History",
        "Computer Applications Technology", "Information Technology", "Engineering Graphics and Design",
        "Agricultural Sciences", "Agricultural Management Practices", "Agricultural Technology",
        "Consumer Studies", "Dramatic Arts", "Dance Studies", "Design", "Music", "Visual Arts",
        "Civil Technology", "Electrical Technology", "Mechanical Technology",
        "Hospitality Studies", "Tourism", "Religion Studies", "Life Orientation"
    ]

    lower_text = text.lower()
    for subject in subjects:
        if subject.lower() in lower_text:
            return subject
    return "Unknown_Subject"

def download_file(url, save_path, filename):
    """
    Downloads a file and saves it to the specified path, with checks for existing files.
    """
    if os.path.exists(save_path):
        print(f"  SKIPPING: File '{filename}' already exists at {save_path}")
        return True # Indicate success as file is already there

    try:
        print(f"  Downloading: {filename} to {os.path.dirname(save_path)}")
        # Use the session object for downloading files as well
        response = session.get(url, stream=True, timeout=30)
        response.raise_for_status()

        with open(save_path, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
        print(f"  SUCCESS: Downloaded '{filename}'")
        return True
    except requests.exceptions.RequestException as e:
        print(f"  ERROR downloading '{filename}' from {url}: {e}")
        return False
    except IOError as e:
        print(f"  ERROR writing file '{save_path}': {e}")
        return False
    except Exception as e:
        print(f"  An unexpected error occurred while downloading '{filename}': {e}")
        return False

# --- Main Scraping Function ---
def scrape_exam_papers():
    """
    Main function to scrape, organize, and download exam papers.
    """
    # Ensure the base Google Drive path exists
    os.makedirs(DRIVE_BASE_PATH, exist_ok=True)
    print(f"Saving files to base directory: {DRIVE_BASE_PATH}")
    print("-" * 70)

    # Fetch the main page content
    html_content = get_page_content(BASE_URL)
    if not html_content:
        print("Failed to retrieve content from the base URL. Exiting.")
        return

    soup = BeautifulSoup(html_content, 'html.parser')
    all_links = soup.find_all('a', href=True)

    download_count = 0
    skipped_count = 0
    total_potential_pdfs = 0

    print(f"Found {len(all_links)} links on the page. Processing...")
    print("-" * 70)

    for i, link in enumerate(all_links):
        href = link['href']
        link_text = link.get_text(strip=True)
        absolute_url = urljoin(BASE_URL, href)

        parsed_url = urlparse(absolute_url)
        filename = os.path.basename(parsed_url.path)

        # For saexampapers.co.za, PDFs are typically linked directly and don't necessarily have
        # '/Portals/0/Documents/' in their path. We'll simplify the check to just look for .pdf extension.
        is_potential_document = (
            filename.lower().endswith('.pdf')
        )

        if is_potential_document:
            total_potential_pdfs += 1
            print(f"\n[{total_potential_pdfs}] Processing link: '{link_text}'")

            combined_text_for_extraction = link_text + " " + filename
            detected_year = extract_year_from_text(combined_text_for_extraction)
            detected_subject = extract_subject_from_text(combined_text_for_extraction)
            doc_type = identify_doc_type(link_text, filename)

            # Filter by configured years
            if TARGET_YEARS and detected_year not in TARGET_YEARS:
                print(f"  SKIPPING: Year {detected_year} not in target years {TARGET_YEARS}.")
                skipped_count += 1
                continue

            # Filter by configured subjects (unless "All" is specified)
            if "All" not in TARGET_SUBJECTS and detected_subject.replace(' ', '').lower() not in [s.replace(' ', '').lower() for s in TARGET_SUBJECTS]:
                print(f"  SKIPPING: Subject '{detected_subject}' not in target subjects {TARGET_SUBJECTS}.")
                skipped_count += 1
                continue

            # Construct the save path
            # Use a default folder if subject/year extraction is ambiguous
            subject_folder = detected_subject if detected_subject != "Unknown_Subject" else "Other_Subjects"
            year_folder = str(detected_year) if detected_year else "Unknown_Year"
            doc_type_folder = doc_type

            final_save_dir = os.path.join(DRIVE_BASE_PATH, subject_folder, year_folder, doc_type_folder)
            os.makedirs(final_save_dir, exist_ok=True) # Ensure directory exists

            full_save_path = os.path.join(final_save_dir, filename)

            print(f"  Identified: Subject='{subject_folder}', Year='{year_folder}', Type='{doc_type_folder}'")
            if download_file(absolute_url, full_save_path, filename):
                download_count += 1
            else:
                skipped_count += 1 # Count download errors as skips too
        # else:
        #     print(f"  Skipping non-relevant link: {absolute_url}") # Uncomment for more verbose output of skipped links

    print("\n" + "=" * 70)
    print("Scraping Summary:")
    print(f"  Total links processed on page: {len(all_links)}")
    print(f"  Total potential PDF links found: {total_potential_pdfs}")
    print(f"  Successfully downloaded: {download_count} files")
    print(f"  Skipped (due to filters, existing, or errors): {skipped_count} files")
    print("=" * 70)

# --- Execute the scraping function ---
scrape_exam_papers()

Saving files to base directory: /content/drive/MyDrive/Matric_Past_Papers
----------------------------------------------------------------------
Fetching URL: https://www.saexampapers.co.za/grade-12-english/
Found 233 links on the page. Processing...
----------------------------------------------------------------------

[1] Processing link: 'English Grade 12 HL Paper 1 - KZNNSC • March 2026 • KwaZulu-Natal'
  Identified: Subject='Other_Subjects', Year='2026', Type='Question_Paper'
  SKIPPING: File 'English-Grade-12-NSC-QP-HL-March-2026-Eng-KZN.pdf' already exists at /content/drive/MyDrive/Matric_Past_Papers/Other_Subjects/2026/Question_Paper/English-Grade-12-NSC-QP-HL-March-2026-Eng-KZN.pdf

[2] Processing link: 'English Grade 12 HL Paper 1 - LPNSC • March 2026 • Limpopo'
  Identified: Subject='Other_Subjects', Year='2026', Type='Question_Paper'
  SKIPPING: File 'English-Grade-12-NSC-QP-HL-March-2026-Eng-LP.pdf' already exists at /content/drive/MyDrive/Matric_Past_Papers/Other_Subject

## Stage 1: Ingestion & Setup

This stage focuses on preparing the environment and building a robust file loader that can handle various input formats (text-based PDFs, scanned PDFs, HTML, or plain text) and normalize them into raw text for subsequent processing.

First, we'll install the required libraries for this stage and beyond.

In [4]:
# Install required libraries for ingestion, text extraction, and NLP stages
# Note: pandas and regex are typically pre-installed in Colab environments.
# PyMuPDF is often imported as 'fitz'.
# !pip install pdfplumber PyMuPDF pytesseract pillow pandas regex
# sentence-transformers langdetect


### Ingesting and Normalizing Raw Data

This section sets up the environment for PDF processing (installing Tesseract OCR) and defines functions to load content from various file types (PDFs, HTML, plain text) into a raw text format. For scanned PDFs, Optical Character Recognition (OCR) will be used.

**Data Location:** Ensure your scraped exam papers are located in the `SOURCE_DATA_PATH` specified below. If you just ran the scraper, they should be in `/content/drive/MyDrive/Matric_Past_Papers/`.

In [5]:
# --- Tesseract OCR Installation (for scanned PDFs) ---
# Tesseract is an external program that pytesseract interfaces with.
!sudo apt update
!sudo apt install tesseract-ocr
!sudo apt install libtesseract-dev

# Configure Tesseract path (often necessary in Colab)
import os
os.environ['TESSDATA_PREFIX'] = '/usr/share/tesseract-ocr/4.00/tessdata' # Adjust if using a different Tesseract version

import pdfplumber
import fitz # PyMuPDF
from PIL import Image
import pytesseract
from io import BytesIO

# --- Configuration for Ingestion ---
# Google Drive base path for saving files. Files will be saved under this path in structured folders.
DRIVE_BASE_PATH = "/content/drive/MyDrive/Matric_Past_Papers"

# Set the path to your raw scraped files (PDFs, HTML, etc.)
# This should match where your scraper saved the files.
SOURCE_DATA_PATH = DRIVE_BASE_PATH # Using the DRIVE_BASE_PATH from the scraper output

# --- File Loading Functions ---

def extract_text_from_pdf_text_layer(pdf_path):
    """
    Extracts text from a PDF document's text layer using pdfplumber.
    Returns a string of all text, or None if extraction fails.
    """
    try:
        with pdfplumber.open(pdf_path) as pdf:
            text_content = []
            for page in pdf.pages:
                page_text = page.extract_text(x_tolerance=2, y_tolerance=2) # Adjust tolerances for better text grouping
                if page_text:
                    text_content.append(page_text)
            return "\n".join(text_content) if text_content else None
    except Exception as e:
        print(f"  Error extracting text layer from {pdf_path}: {e}")
        return None

def preprocess_image_for_ocr(image):
    """
    Applies basic image preprocessing for OCR (grayscale, thresholding, deskew).
    """
    # Convert to grayscale
    image = image.convert('L')

    # Apply thresholding
    image = image.point(lambda x: 0 if x < 128 else 255, '1')

    # (Optional) Deskewing can be complex and might require external libraries or more advanced algorithms.
    # For simplicity, we'll skip sophisticated deskewing for now.
    # from deskew import determine_skew
    # from skimage.transform import rotate
    # angle = determine_skew(np.array(image))
    # if angle is not None:
    #     image = Image.fromarray(rotate(np.array(image), angle, resize=True, mode='constant', cval=255) * 255).convert('1')

    # (Optional) Denoising. For binary images, a simple median filter might work.
    # from scipy.ndimage import median_filter
    # image = Image.fromarray(median_filter(np.array(image), size=3)).convert('1')

    return image

def extract_text_from_scanned_pdf_with_ocr(pdf_path):
    """
    Extracts text from a scanned PDF using PyMuPDF (fitz) to get images and pytesseract for OCR.
    Applies basic image preprocessing.
    Returns a string of all OCR'd text, or None if extraction fails.
    """
    try:
        doc = fitz.open(pdf_path)
        full_text = []
        for page_num in range(len(doc)):
            page = doc.load_page(page_num)
            pix = page.get_pixmap()
            img_bytes = pix.tobytes("png") # Get PNG image bytes
            img = Image.open(BytesIO(img_bytes))

            # Preprocess image
            processed_img = preprocess_image_for_ocr(img)

            # Perform OCR
            page_text = pytesseract.image_to_string(processed_img, lang='eng') # Specify language
            if page_text:
                full_text.append(page_text)
        doc.close()
        return "\n".join(full_text) if full_text else None
    except Exception as e:
        print(f"  Error performing OCR on {pdf_path}: {e}")
        return None

def extract_text_from_html(html_path):
    """
    Extracts text from an HTML file using BeautifulSoup.
    """
    try:
        with open(html_path, 'r', encoding='utf-8') as f:
            # Import BeautifulSoup locally to avoid circular dependency if it's not globally available from a prior cell
            from bs4 import BeautifulSoup
            soup = BeautifulSoup(f, 'html.parser')
            # Get text from common content tags, ignoring script/style
            for script_or_style in soup(['script', 'style']):
                script_or_style.extract()
            text = soup.get_text(separator='\n', strip=True)
            return text
    except Exception as e:
        print(f"  Error extracting text from HTML {html_path}: {e}")
        return None

def extract_text_from_plain_file(file_path):
    """
    Extracts text from a plain text file.
    """
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            return f.read()
    except Exception as e:
        print(f"  Error reading plain text file {file_path}: {e}")
        return None

def load_document_to_raw_text(file_path, enable_ocr=True):
    """
    Loads a document (PDF, HTML, TXT) and returns its raw text content.
    Attempts text layer extraction first for PDFs, then OCR if enable_ocr is True.
    """
    print(f"Loading document: {file_path}")
    file_extension = os.path.splitext(file_path)[1].lower()

    text_content = None

    if file_extension == '.pdf':
        # Try text layer extraction first
        text_content = extract_text_from_pdf_text_layer(file_path)
        if text_content:
            print("  Successfully extracted text from PDF text layer.")
        elif enable_ocr:
            print("  No text layer found or extraction failed, attempting OCR...")
            text_content = extract_text_from_scanned_pdf_with_ocr(file_path)
            if text_content:
                print("  Successfully extracted text via OCR.")
            else:
                print("  OCR also failed.")
        else:
            print("  No text layer found and OCR is disabled.")
    elif file_extension == '.html' or file_extension == '.htm':
        text_content = extract_text_from_html(file_path)
        if text_content: print("  Successfully extracted text from HTML.")
    elif file_extension == '.txt':
        text_content = extract_text_from_plain_file(file_path)
        if text_content: print("  Successfully extracted text from plain text file.")
    else:
        print(f"  Unsupported file type: {file_extension}. Skipping.")

    if text_content is None:
        print(f"  WARNING: Could not extract any text from {file_path}.")

    return text_content


Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Get:3 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Get:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 24.6 kB in 3s (8,400 B/s)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
86 packages can be upgraded. Run 'apt list --upgradable' to see them.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


## Stage 2: Ingesting Raw Documents into Structured Data

This stage involves traversing the `SOURCE_DATA_PATH` (where the scraped PDF, HTML, and text files are stored), and using the `load_document_to_raw_text` function to extract the content of each relevant file. The extracted text, along with metadata (like file path, subject, year, document type), will be stored in a list of dictionaries, forming our initial raw dataset. We'll also define helper functions to extract metadata from the file paths.

### Metadata Extraction from File Paths

To enrich our data, we'll extract `subject`, `year`, and `document_type` directly from the hierarchical folder structure created by the scraper. This ensures consistent metadata for each document.

In [6]:
def extract_metadata_from_path(file_path):
    """
    Extracts subject, year, and document type from the file path structure.
    Assumes a structure like: /path/to/Matric_Past_Papers/Subject/Year/Document_Type/filename.pdf
    """
    parts = file_path.split(os.sep)
    # Find the index of 'Matric_Past_Papers' to get relative path parts
    try:
        base_path_index = parts.index(DRIVE_BASE_PATH.split(os.sep)[-1]) # Get the last part of DRIVE_BASE_PATH
        # Subject, Year, DocType should follow the base path
        subject = parts[base_path_index + 1] if base_path_index + 1 < len(parts) else "Unknown_Subject"
        year = parts[base_path_index + 2] if base_path_index + 2 < len(parts) else "Unknown_Year"
        doc_type = parts[base_path_index + 3] if base_path_index + 3 < len(parts) else "Unknown_Document_Type"

        # Convert year to int if possible
        try:
            year = int(year)
        except ValueError:
            year = "Unknown_Year"

        return {
            'subject': subject,
            'year': year,
            'document_type': doc_type,
        }
    except ValueError:
        # DRIVE_BASE_PATH part not found in file_path, fall back to default unknowns
        print(f"Warning: Could not find '{DRIVE_BASE_PATH}' in path: {file_path}. Using default metadata.")
        return {
            'subject': "Unknown_Subject",
            'year': "Unknown_Year",
            'document_type': "Unknown_Document_Type",
        }

def ingest_documents_to_raw_data(source_dir, enable_ocr=True):
    """
    Traverses the source directory, loads relevant documents, and collects raw text
    and metadata into a list of dictionaries.
    """
    raw_documents = []
    for root, _, files in os.walk(source_dir):
        for file_name in files:
            file_path = os.path.join(root, file_name)
            # Only process files that are likely exam papers (PDF, HTML, TXT)
            if file_name.lower().endswith(('.pdf', '.html', '.htm', '.txt')):
                print(f"Processing file: {file_path}")
                metadata = extract_metadata_from_path(file_path)
                text_content = load_document_to_raw_text(file_path, enable_ocr=enable_ocr)

                if text_content:
                    raw_documents.append({
                        'file_path': file_path,
                        'filename': file_name,
                        'raw_text': text_content,
                        **metadata # Unpack metadata dictionary
                    })
    return raw_documents

### Run Ingestion and Create Initial DataFrame

Now, let's execute the ingestion process. This will iterate through all the files in your scraped directory, extract their raw text, gather metadata, and store everything in a pandas DataFrame for easier manipulation in subsequent steps.

In [7]:
import os
import pandas as pd
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Define path on Drive (adjust to your folder structure)
CACHE_FILE_PATH = '/content/drive/MyDrive/processed_raw_documents.parquet'

# 3. Check if cached data exists
if os.path.exists(CACHE_FILE_PATH):
    print(f"Cache found! Loading ingested data directly from Drive...")
    df_raw_documents = pd.read_parquet(CACHE_FILE_PATH)
else:
    print(f"No cache found. Starting ingestion from: {SOURCE_DATA_PATH}")
    initial_raw_data = ingest_documents_to_raw_data(SOURCE_DATA_PATH, enable_ocr=True)
    print(f"Finished ingestion. Total documents processed: {len(initial_raw_data)}")

    df_raw_documents = pd.DataFrame(initial_raw_data)

    # Save to Drive for future sessions
    df_raw_documents.to_parquet(CACHE_FILE_PATH)
    print(f"Saved processed data to Drive: {CACHE_FILE_PATH}")

# 4. Display details
print("\nRaw Documents DataFrame Info:")
df_raw_documents.info()
display(df_raw_documents.head())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Cache found! Loading ingested data directly from Drive...

Raw Documents DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 168 entries, 0 to 167
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   file_path      168 non-null    object
 1   filename       168 non-null    object
 2   raw_text       168 non-null    object
 3   subject        168 non-null    object
 4   year           168 non-null    int64 
 5   document_type  168 non-null    object
dtypes: int64(1), object(5)
memory usage: 8.0+ KB


,file_path,filename,raw_text,subject,year,document_type
0,/content/drive/MyDrive/Matric_Past_Papers/Othe...,English-Grade-12-NSC-QP-HL-March-2026-Eng-KZN.pdf,SA EXAM PAPERS | This past paper was downloade...,Other_Subjects,2026,Question_Paper
1,/content/drive/MyDrive/Matric_Past_Papers/Othe...,English-Grade-12-NSC-QP-HL-March-2026-Eng-LP.pdf,SA EXAM PAPERS | This past paper was downloade...,Other_Subjects,2026,Question_Paper
2,/content/drive/MyDrive/Matric_Past_Papers/Othe...,English-Grade-12-NSC-HL-P1-QP-May-June-2026.pdf,Confidential\nSENIOR CERTIFICATE EXAMINATIONS/...,Other_Subjects,2026,Question_Paper
3,/content/drive/MyDrive/Matric_Past_Papers/Othe...,English-Grade-12-NSC-HL-P2-QP-May-June-2026.pdf,Confidential\nSENIOR CERTIFICATE EXAMINATIONS/...,Other_Subjects,2026,Question_Paper
4,/content/drive/MyDrive/Matric_Past_Papers/Othe...,English-Grade-12-NSC-HL-P3-QP-May-June-2026.pdf,Confidential\nSENIOR CERTIFICATE EXAMINATIONS/...,Other_Subjects,2026,Question_Paper


## Stage 2: Text Cleaning and Preprocessing

This stage focuses on refining the extracted raw text by removing noise and inconsistencies that often appear in scanned or converted documents. The goal is to obtain a clean, normalized text representation suitable for downstream NLP tasks.

Key cleaning steps will include:
1.  **Normalize Whitespace**: Remove excessive spaces, tabs, and newlines.
2.  **Remove Page Headers/Footers**: Attempt to identify and remove repeating header/footer patterns.
3.  **Remove Page Numbers**: Identify and remove numerical page indicators.
4.  **Remove Boilerplate/Watermarks**: Remove common disclaimers or watermarks specific to the source (e.g., from `saexampapers.co.za`).
5.  **Remove Short/Non-Informative Lines**: Filter out very short lines that are likely noise.

We'll apply these cleaning functions to the `raw_text` column and store the result in a new `cleaned_text` column.

In [8]:
import re
import numpy as np

def normalize_whitespace(text):
    """
    Removes excessive whitespace and standardizes newlines.
    Preserves single newlines to maintain line structure, but cleans leading/trailing
    whitespace from each line and collapses multiple empty lines.
    """
    if not isinstance(text, str): return text

    # Split text into lines, strip leading/trailing whitespace from each line,
    # and replace multiple internal spaces with single spaces.
    cleaned_lines = [re.sub(r'[ \t]+', ' ', line.strip()) for line in text.split('\n')]

    # Remove completely empty lines, but keep lines that contain only whitespace
    # if stripping made them empty.
    # Rejoin with single newlines, then collapse excessive blank lines.
    text_with_single_newlines = '\n'.join(line for line in cleaned_lines if line) # Filter out truly empty lines

    # Collapse three or more consecutive newlines into two newlines
    text = re.sub(r'\n{3,}', '\n\n', text_with_single_newlines)

    # Final strip of overall leading/trailing whitespace
    return text.strip()

def remove_page_numbers(text):
    """
    Removes isolated page numbers, often found at the top or bottom of pages.
    Looks for numbers or numbers within brackets/parentheses at line start/end.
    """
    if not isinstance(text, str): return text
    # Pattern to find isolated numbers, possibly with hyphens, slashes, or in parentheses/brackets
    # at the start/end of a line or surrounded by spaces.
    # Example: '1', 'Page 1', '1 of 10', '(1)', '[1]' -- but be careful not to remove years/values
    lines = text.split('\n')
    cleaned_lines = []
    for line in lines:
        # More robust page number detection: at start or end of line, possibly with 'Page'
        if re.fullmatch(r'\s*\b(page|bladsy)?\s*\d+\s*(of\s*\d+)?\b\s*', line.lower().strip(), re.IGNORECASE) or \
           re.fullmatch(r'\s*(\[\s*|\()?\d+(\s*|/\s*\d+)(\s*|]|-|\))\s*', line.strip()): # Added '-' to pattern
            continue # Skip line if it looks like a page number line
        cleaned_lines.append(line)
    return '\n'.join(cleaned_lines)

def remove_boilerplate(text):
    """
    Removes specific boilerplate text or watermarks.
    Customized for 'saexampapers.co.za' as seen in initial data, and common exam paper elements.
    """
    if not isinstance(text, str): return text

    boilerplate_patterns = [
        # SA Exam Papers specific download messages and unique characters (Corrected escaping of . and |)
        # Using more robust patterns to ensure removal of full lines.
        r'^.*SA EXAM PAPERS \| This past paper was downloaded from saexampapers\.co\.za.*$', # Catch full line including variable end
        r'^.*You have Downloaded, yet Another Great.*Resource to assist you with your Studies.*$', # Catch the long download message
        r'^.*Thank You for Supporting SA Exam Papers.*$',
        r'^.*Your Leading Past Year Exam Paper Resource Portal.*$',
        r'^.*Visit us @ www\.saexampapers\.co\.za.*$',
        r'^.*Please visit saexampapers\.co\.za for more resources\..*$',
        r'www\.saexampapers\.co\.za', # Redundant if caught by above but good fallback
        r'This past paper was downloaded from saexampapers\.co\.za', # Redundant if caught by above but good fallback
        r'\uf04a', # Specific unicode character, often present with download messages
        r'\(cid:\d+\)', # Remove (cid:X) patterns from PDF extraction

        # General Exam Paper Headers/Footers/Instructions - made more flexible and comprehensive
        # The previous combined regex failed if the header was on multiple lines.
        r'^Confidential$',
        r'^SENIOR CERTIFICATE EXAMINATIONS/?$',
        r'^NATIONAL SENIOR CERTIFICATE EXAMINATIONS$',
        r'^ENGLISH HOME LANGUAGE P[1-3]$', # Match full line for subject header
        r'^MAY/JUNE \d{4}$', # Match full line for month/year
        r'^MARKS: \d+$', # Match full line for marks
        r'^TIME: \d+ hours$', # Match full line for time
        r'^This (?:question paper )?consists of \d+ pages\.(?: Copyright reserved)?$', # Make 'Copyright reserved' optional
        r'^Copyright reserved Please turn over$', # Specific combined line
        r'^Copyright reserved$', # Standalone copyright
        r'^Please turn over$', # Standalone please turn over
        # More robust patterns for page footers:
        r'^English Home Language/P[1-3]\s*\d+\s*DBE/.*$', # Modified: Catch first part of footer, more flexible with spacing
        r'^DBE/(?:MAY/JUNE|NOVEMBER) \d{4} SC/NSC(?: Confidential)?$', # Catch if "English Home Language" part is missing
        r'^SC/NSC Confidential$', # If only this part remains
        r'^INSTRUCTIONS AND INFORMATION$', # Specific header, now with anchors
        r'^QUESTION PAPER$', # "QUESTION PAPER" often appears as a header, now with anchors
        r'^Annexure$', # Often found in addendums, now with anchors
        r'^This consists of \d+ pages\.$', # Catches the specific 'This consists of 12 pages.' left behind (now without 'Copyright reserved' for flexibility)

        # Flexible instruction phrases - combined and made more robust
        r'Read ALL the instructions carefully', # Specific instruction
        r'Answer ALL the questions', # Specific instruction
        r'Start EACH section on a NEW page', # Specific instruction
        r'Rule off after each section', # Specific instruction
        r'Number the answers correctly according to the numbering system used in this(?: question paper)?', # Specific instruction
        r'Leave a line after each answer', # Specific instruction
        r'Pay special attention to spelling and sentence construction', # Specific instruction
        r'Suggested time allocation:', # Specific instruction
        r'Write neatly and legibly', # Specific instruction
        r'^(?:\s*\d+\.\s*)?This\s+(?:question paper\s+)?consists of (?:THREE|TWO|FOUR|\d+)(?: sections)?:.*$', # Modified: More robust for "This question paper consists of X sections:"
        r'You must write a fluent paragraph', # Specific instruction
        r'You are NOT required to include a title(?: for the \.)?', # Specific instruction

        r'SECTIONS? [A-C]:\s*(?:\d+\s+minutes|[A-Z ]+)?(?:\(\d+\))?', # Section headers

        # Section Headers (more flexible, including content description)
        r'SECTION [A-Z](?::\s*[A-Z0-9 ]+)?\s*(?:\(\d+\))?',
        r'SECTION [A-Z]:\s*(?:\d+\s+minutes|[A-Z ]+)',
        r'TEXT [A-Z](?:\s+[A-Z_ ]+)?',
        r'Read TEXTS [A-Z] and [A-Z] below and answer the questions set\.',
        r'READING FOR MEANING AND UNDERSTANDING',
        r'COMPREHENSION',
        r'SUMMARY',
        r'LANGUAGE STRUCTURES AND CONVENTIONS',

        # Specific instruction lines (made more flexible for leading dots/spaces/newlines)
        r'^(?:\s*\.{2,}\s*)$', # Matches lines with just dots, e.g., '. . . .'
        r'\s*\d+\\.\s*$', # Catches lone numbers like '1.' on a line if they're not question headers
        r'^–$', # Matches standalone em-dash often used as bullet
        r'^—$', # Matches standalone en-dash often used as bullet
        r'^•$', # Matches standalone bullet point
        r'^●$', # Matches standalone bullet point
        r'^○$', # Matches standalone circle bullet point
        r'^\s*question paper\.\s*$', # Added: Specifically target the 'question paper.' artifact
        r'^–$', # Matches standalone em-dash often used as bullet
        r'^—$', # Matches standalone en-dash often used as bullet
        r'^•$', # Matches standalone bullet point
        r'^●$', # Matches standalone bullet point
        r'^○$', # Matches standalone circle bullet point
        r'^–$', # Matches standalone em-dash often used as bullet
        r'^—$', # Matches standalone en-dash often used as bullet
        r'^•$', # Matches standalone bullet point
        r'^●$', # Matches standalone bullet point
        r'^○$', # Matches standalone circle bullet point
        r'^–$', # Matches standalone em-dash often used as bullet
        r'^—$', # Matches standalone en-dash often used as bullet
        r'^•$', # Matches standalone bullet point
        r'^●$', # Matches standalone bullet point
        r'^○$', # Matches standalone circle bullet point
        r'^–$', # Matches standalone em-dash often used as bullet
        r'^—$', # Matches standalone en-dash often used as bullet
        r'^•$', # Matches standalone bullet point
        r'^●$', # Matches standalone bullet point
        r'^○$', # Matches standalone circle bullet point
        r'^\s*\d+\s*$', # Catches lone numbers on a line
        r'^\d{4}$', # Catches standalone 4-digit year like '2026' on a line
        r'^\[\s*\d+\s*\]$', # Matches patterns like '[ 1 ]' which are often page numbers or score indicators

        # Scraping summary text (should not be in documents, but defensive)
        r'Total potential PDF links found: \d+',
        r'Successfully downloaded: \d+ files',
        r'Skipped \(due to filters, existing, or errors\): \d+ files'
    ]

    # Remove redundant unicode bullet patterns before processing
    # The list contains many duplicates now, remove them here if they appear multiple times.
    boilerplate_patterns = list(dict.fromkeys(boilerplate_patterns))

    for pattern in boilerplate_patterns:
        text = re.sub(pattern, '', text, flags=re.IGNORECASE | re.MULTILINE)
    return text

def remove_short_lines(text, min_length=20):
    """
    Removes lines shorter than min_length (after stripping) that are unlikely to be informative.
    """
    if not isinstance(text, str): return text
    lines = text.split('\n')
    cleaned_lines = [line for line in lines if len(line.strip()) >= min_length]
    return '\n'.join(cleaned_lines)

def clean_text_pipeline(text):
    """
    Applies a sequence of cleaning steps to the raw text.
    Revised order: page numbers, normalize whitespace, boilerplate, then short lines.
    """
    if not isinstance(text, str): return text

    text = remove_page_numbers(text) # Should be applied before normalize_whitespace for better line detection
    text = normalize_whitespace(text)
    text = remove_boilerplate(text) # Now boilerplate removal will work better with proper lines
    text = remove_short_lines(text, min_length=15) # Adjust min_length as needed
    return text

# Apply the cleaning pipeline to the raw_text column
print("Applying text cleaning pipeline...")
df_raw_documents['cleaned_text'] = df_raw_documents['raw_text'].apply(clean_text_pipeline)
print("Text cleaning complete.")

# Display the difference between raw and cleaned text for a sample document
print("\nSample of raw vs. cleaned text (first document):")
sample_index = 0
# Find a Question_Paper document that is not empty for better debugging
sample_doc = df_raw_documents[(df_raw_documents['document_type'] == 'Question_Paper') & (df_raw_documents['cleaned_text'].str.len() > 500)].iloc[0] if not df_raw_documents[(df_raw_documents['document_type'] == 'Question_Paper') & (df_raw_documents['cleaned_text'].str.len() > 500)].empty else None

if sample_doc is not None:
    print("--- Raw Text ---")
    print(sample_doc['raw_text'][:500]) # Display first 500 chars
    print("\n--- Cleaned Text ---")
    print(sample_doc['cleaned_text'][:500]) # Display first 500 chars
else:
    print("No suitable 'Question_Paper' document found with substantial cleaned text for sampling.")

# Check for empty cleaned texts (documents that became entirely noise)
empty_cleaned_docs = df_raw_documents[df_raw_documents['cleaned_text'].apply(lambda x: not x or len(x.strip()) < 50)]
if not empty_cleaned_docs.empty:
    print(f"\nWARNING: {len(empty_cleaned_docs)} documents resulted in very short or empty cleaned text. Review them:\n")
    display(empty_cleaned_docs[['file_path', 'filename', 'raw_text', 'cleaned_text']])
else:
    print("\nAll documents retained substantial content after cleaning.")

Applying text cleaning pipeline...
Text cleaning complete.

Sample of raw vs. cleaned text (first document):
--- Raw Text ---
Confidential
SENIOR CERTIFICATE EXAMINATIONS/
NATIONAL SENIOR CERTIFICATE EXAMINATIONS
ENGLISH HOME LANGUAGE P1
MAY/JUNE 2026
MARKS: 70
TIME: 2 hours
This question paper consists of 12 pages.
Copyright reserved Please turn over
English Home Language/P1 2 DBE/May/June 2026
SC/NSC Confidential
INSTRUCTIONS AND INFORMATION
2026
1. This question paper consists of THREE sections:
SECTION A: Comprehension (30)
SECTION B: Summary (10)
SECTION C: Language structures and conventions (30)
2. Read ALL the 

--- Cleaned Text ---
1 Imagine the vast spectrum of all the cultures in the world. Listen to the music –
from the gentle drum beats of Africa, to the scream of the electric guitar. Taste the
curry from India, the coconut milk from Thailand, the cheeseburger from the
United States. Now imagine that all these cultures are compressed into one super-
culture creating a glob

## Stage 3: Question Segmentation

This stage is crucial for transforming raw document text into a structured question bank. We will develop a parser to identify and segment individual questions within each cleaned document. This involves:

1.  **Identifying Question Patterns**: Using regular expressions to detect common question numbering schemes (e.g., "QUESTION 1", "1.", "1.1").
2.  **Extracting Question Text**: Grouping the lines of text that belong to each identified question.
3.  **Handling Multi-part Questions**: The current approach will try to keep sub-parts (e.g., 1.1, 1.2) together under their main question if a top-level question header is present, or segment them as individual questions if they are the primary numbering scheme.

The output of this stage will be a new column in our DataFrame, containing a list of dictionaries, where each dictionary represents a segmented question with its identifier and text.

In [9]:
import re
import pandas as pd

def segment_questions(text):
    """
    Segments a document's cleaned text into individual questions based on common patterns.
    Returns a list of dictionaries, where each dictionary represents a question
    and includes its identifier (e.g., "QUESTION 1", "1.1") and text.
    """
    if not isinstance(text, str) or not text.strip():
        return []

    questions = []
    current_question_text = []
    current_question_id = None

    # Refined Regex to detect major question numbers and sub-question numbers.
    # It now specifically looks for actual numerical questions like '1.', '1.1.', '1)' or '1.1)'
    # and prioritizes them over section headers like 'QUESTION X'.
    question_start_pattern = re.compile(r"""^( # Match at the beginning of a line
    (?P<full_id>
        # Pattern for "1. Question text" or "1.1. Question text"
        # Must be followed by a space and an uppercase letter or an opening quote.
        # This ensures we don't accidentally match general instructions like '1. This consists...'
        \d+(?:\.\d+)*\.\s*(?=[A-Z"'\u201c\u2018])
        |              # OR
        # Pattern for "1) Question text" or "1.1) Question text"
        \d+(?:\.\d+)*\)\s*(?=[A-Z"'\u201c\u2018])
        # Removed 'QUESTION X' from here, as it's now considered preamble.
    )
)""", re.MULTILINE | re.VERBOSE | re.IGNORECASE)

    # Pattern to identify content that is likely preamble or section headers that should be skipped
    # This pattern is now primarily for debugging or for future, more nuanced filtering.
    # It is NOT used to filter lines within the main segmentation loop anymore.
    preamble_pattern = re.compile(r"""^(?: # Match at the beginning of a line
        # High-level question/section markers
        \s*QUESTION\s*\d+(?:\.\d+)*:?.* # e.g., 'QUESTION 2', 'QUESTION 3: ANALYSING ADVERTISING'
        |\s*SECTION\s*[A-Z].* # e.g., 'SECTION A: COMPREHENSION'
        # General instructions/preamble that are not questions
        |\s*:\s*(?:SUMMARISING|ANALYSE|UNDERSTANDING|USING)\s+.* # e.g., ': SUMMARISING IN YOUR OWN WORDS'
        |\s*(?:STUDY|READ|INSTRUCTIONS|NOTE|REFER TO):?\s*.* # e.g., 'Study the advertisement', 'Read TEXT A', 'Refer to TEXT B'
        |\s*FRAME\s*\d+\s*.* # e.g., 'FRAME 1', 'FRAME 2.1'
        # Numbered instructions that look like question IDs but are not actual questions
        |\s*\d+(?:\.\d+)*[\.\)]\s*(?:READ|STUDY|REFER TO|CONSIDER|ANALYSE|SUMMARISE|EXPLAIN|DISCUSS|WRITE|PROVIDE|IDENTIFY|STATE|COMMENT ON|INDICATE)\.?.* # More robust numbered instructions
        # Passage titles or specific descriptive lines (flexible to leading non-alphanum)
        |\s*(?:MAIN SPEAKER|CHARACTER IN THE RIGHT-HAND CORNER OF THE LAST FRAME|TEXT\s*[A-Z])(?::\s*.*)? # Made colon optional for these
        |\s*(?:\.{3}|\u2026|\W*?)?\s*A FAVOURABLE ALTERNATIVE.* # Robustly match the passage title, including '...' or '…'
        |\s*What if improving your physical and mental health was as easy as riding in a taxi, bus or train\??.* # Specific passage start line
        |\s*Breathing fresh air, doing physical activity and avoiding stress are a few well-known steps toward healthy living\..* # Specific passage line
        |\s*Individuals who.* # Specific passage line
        |\s*Indicate your word count.* # Specific instruction phrase
        |\s*\.\s*$ # Matches lines with just dots/periods
    )$""", re.MULTILINE | re.VERBOSE | re.IGNORECASE)

    lines = text.split('\n')
    for line in lines:
        line_stripped = line.strip()
        if not line_stripped: # Skip empty lines after stripping
            continue

        # --- IMPORTANT CHANGE --- Removed line filtering by preamble_pattern here.
        # Per user instructions, instructions and context should be kept with questions for RAG.

        match = question_start_pattern.match(line_stripped)

        if match:
            # If we found a new question ID, save the previous one if it exists
            if current_question_id and current_question_text:
                questions.append({
                    'question_id': current_question_id,
                    'question_text': '\n'.join([t for t in current_question_text if t.strip()]).strip()
                })
            # Start new question
            current_question_id = match.group('full_id').strip() # Capture the specific ID
            # Remove the matched ID from the line to avoid duplication in text
            remaining_text = line_stripped[match.end():].strip()

            # --- IMPORTANT CHANGE --- Removed instruction_verbs_pattern filtering.
            # All remaining text after the ID is considered part of the question's content/context.
            if remaining_text:
                current_question_text = [remaining_text] # Start text with the current line after ID
            else:
                current_question_text = [] # No remaining text after ID
        else:
            # If no new question ID, append to current question's text
            # Only append if a question is currently being built
            if current_question_id:
                current_question_text.append(line_stripped)

    # Add the last question
    if current_question_id and current_question_text:
        questions.append({
            'question_id': current_question_id,
            'question_text': '\n'.join([t for t in current_question_text if t.strip()]).strip()
        })

    return questions

# Apply the question segmentation pipeline to the 'cleaned_text' column
print("Applying question segmentation pipeline...")
df_raw_documents['segmented_questions'] = df_raw_documents['cleaned_text'].apply(segment_questions)
print("Question segmentation complete.")

# Display the results for a sample document that is a Question_Paper AND has segmented questions
print("\nSample of segmented questions (first Question Paper document with content AND segmented questions):")
sample_doc_with_questions = None
for idx, row in df_raw_documents.iterrows():
    if row['document_type'] == 'Question_Paper' and row['cleaned_text'] and row['segmented_questions']:
        sample_doc_with_questions = row
        break

if sample_doc_with_questions is not None:
    print(f"File: {sample_doc_with_questions['filename']}")
    print(f"Subject: {sample_doc_with_questions['subject']}, Year: {sample_doc_with_questions['year']}, Type: {sample_doc_with_questions['document_type']}")
    print("\n--- Segmented Questions ---")
    for i, question in enumerate(sample_doc_with_questions['segmented_questions'][:5]): # Display first 5 questions
        print(f"Question {i+1} (ID: {question['question_id']}):\n{question['question_text'][:300]}...") # Truncate for display
        print("\n" + "-"*20)
    if len(sample_doc_with_questions['segmented_questions']) > 5:
        print(f"... and {len(sample_doc_with_questions['segmented_questions']) - 5} more questions.")
else:
    print("Could not find a suitable 'Question_Paper' document with substantial cleaned text and segmented questions for sampling.")
    # If no segmented questions found in any suitable document, show the cleaned_text of the first non-empty one for debugging.
    first_non_empty_cleaned_text_doc = df_raw_documents[(df_raw_documents['document_type'] == 'Question_Paper') & (df_raw_documents['cleaned_text'].str.len() > 500)].iloc[0] if not df_raw_documents[(df_raw_documents['document_type'] == 'Question_Paper') & (df_raw_documents['cleaned_text'].str.len() > 500)].empty else None
    if first_non_empty_cleaned_text_doc is not None:
        print(f"\nDisplaying cleaned text for debugging from: {first_non_empty_cleaned_text_doc['filename']}")
        print(first_non_empty_cleaned_text_doc['cleaned_text'][:1000]) # Print first 1000 chars for inspection

# Display general info about the new column
print("\nDataFrame Info after segmentation:")
df_raw_documents.info()

Applying question segmentation pipeline...
Question segmentation complete.

Sample of segmented questions (first Question Paper document with content AND segmented questions):
File: English-Grade-12-NSC-HL-P1-QP-May-June-2026.pdf
Subject: Other_Subjects, Year: 2026, Type: Question_Paper

--- Segmented Questions ---
Question 1 (ID: 4.):
Indicate your word count at the end of your .
… A FAVOURABLE ALTERNATIVE
What if improving your physical and mental health was as easy as riding in a taxi, bus
or train? Breathing fresh air, doing physical activity and avoiding stress are a few
well-known steps toward healthy living.
Individuals who...

--------------------

DataFrame Info after segmentation:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 168 entries, 0 to 167
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   file_path            168 non-null    object
 1   filename             168 non-null    object


## Stage 4: Metadata Enrichment & Tagging

This stage focuses on extracting further semantic metadata from the segmented questions themselves. This will involve:

1.  **Identifying Question Type**: Classifying questions into categories like 'Definition', 'Explanation', 'Analysis', 'Discussion', 'Identification', 'Application', 'Summary', etc.
2.  **Extracting Keywords/Topics**: Identifying key terms or topics discussed within each question.
3.  **(Future)** **Assessing Difficulty/Cognitive Level**: Potentially using NLP techniques to estimate the complexity of questions.

The goal is to enrich each segmented question with structured tags that can be used for advanced search, filtering, and analysis in the final question bank. We'll start with identifying basic question types.

In [12]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Download necessary NLTK data (if not already downloaded)
try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')
try:
    # Sometimes 'word_tokenize' implicitly requires 'punkt_tab'
    nltk.data.find('tokenizers/punkt_tab')
except LookupError:
    nltk.download('punkt_tab')

# Define common stop words
stop_words = set(stopwords.words('english'))

def get_question_type(question_text):
    """
    Analyzes a question's text to determine its likely type based on keywords and patterns.
    """
    lower_text = question_text.lower()

    if re.search(r'define|what is|what are|meaning of', lower_text):
        return 'Definition'
    elif re.search(r'explain|describe|how to|in your own words', lower_text):
        return 'Explanation/Description'
    elif re.search(r'analyze|analyse|discuss|evaluate|critique|examine', lower_text):
        return 'Analysis/Discussion'
    elif re.search(r'identify|list|name|state|mention', lower_text):
        return 'Identification/Listing'
    elif re.search(r'compare|contrast', lower_text):
        return 'Comparison'
    elif re.search(r'suggest|propose|recommend|advise', lower_text):
        return 'Suggestion/Recommendation'
    elif re.search(r'interpret|infer| deduce|imply', lower_text):
        return 'Interpretation'
    elif re.search(r'provide evidence|justify|support|substantiate', lower_text):
        return 'Justification/Evidence'
    elif re.search(r'summarise|summarize', lower_text):
        return 'Summary'
    elif re.search(r'apply|demonstrate|illustrate', lower_text):
        return 'Application/Demonstration'
    elif re.search(r'calculate|determine|find', lower_text):
        return 'Calculation/Determination'
    elif re.search(r'give a reason|account for', lower_text):
        return 'Reasoning'
    elif re.search(r'choose|select', lower_text):
        return 'Choice/Selection'
    elif re.search(r'correct the error|rewrite the sentence|rephrase', lower_text):
        return 'Correction/Rewriting'
    elif re.search(r'true or false|yes or no', lower_text):
        return 'True/False/Yes/No'
    elif re.search(r'which of the following', lower_text):
        return 'Multiple Choice (Implicit)'
    return 'General/Other'

def extract_keywords(text, num_keywords=5):
    """
    Extracts key terms from the question text using simple tokenization and stop word removal.
    """
    if not isinstance(text, str): return []
    words = word_tokenize(text.lower())
    filtered_words = [word for word in words if word.isalnum() and word not in stop_words]
    # For a simple approach, just return the most frequent non-stopwords, or unique words.
    # More sophisticated methods (TF-IDF, RAKE, TextRank) would be used in a production system.
    from collections import Counter
    word_counts = Counter(filtered_words)
    return [word for word, count in word_counts.most_common(num_keywords)]

# Function to apply enrichment to segmented questions
def enrich_segmented_questions(segmented_questions_list):
    enriched_list = []
    if not isinstance(segmented_questions_list, list): return enriched_list

    for question in segmented_questions_list:
        q_text = question.get('question_text', '')
        question['question_type'] = get_question_type(q_text)
        question['keywords'] = extract_keywords(q_text, num_keywords=5) # Extract top 5 keywords
        enriched_list.append(question)
    return enriched_list

# Apply the enrichment pipeline to the 'segmented_questions' column
print("Applying metadata enrichment to segmented questions...")
df_raw_documents['segmented_questions'] = df_raw_documents['segmented_questions'].apply(enrich_segmented_questions)
print("Metadata enrichment complete.")

# Display updated sample of segmented questions
print("\nSample of enriched segmented questions (first Question Paper document):")
sample_doc_enriched = None
for idx, row in df_raw_documents.iterrows():
    if row['document_type'] == 'Question_Paper' and row['segmented_questions']:
        sample_doc_enriched = row
        break

if sample_doc_enriched is not None:
    print(f"File: {sample_doc_enriched['filename']}")
    print(f"Subject: {sample_doc_enriched['subject']}, Year: {sample_doc_enriched['year']}, Type: {sample_doc_enriched['document_type']}")
    print("\n--- Enriched Segmented Questions ---")
    for i, question in enumerate(sample_doc_enriched['segmented_questions'][:3]): # Display first 3 enriched questions
        print(f"Question {i+1} (ID: {question['question_id']}):")
        print(f"  Type: {question['question_type']}")
        print(f"  Keywords: {', '.join(question['keywords'])}")
        print(f"  Text: {question['question_text'][:200]}...") # Truncate for display
        print("\n" + "-"*20)
    if len(sample_doc_enriched['segmented_questions']) > 3:
        print(f"... and {len(sample_doc_enriched['segmented_questions']) - 3} more enriched questions.")
else:
    print("Could not find a suitable 'Question_Paper' document with segmented questions for sampling after enrichment.")

# Display general info about the new column to confirm structure
print("\nDataFrame Info after enrichment:")
df_raw_documents.info()

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Applying metadata enrichment to segmented questions...
Metadata enrichment complete.

Sample of enriched segmented questions (first Question Paper document):
File: English-Grade-12-NSC-HL-P1-QP-May-June-2026.pdf
Subject: Other_Subjects, Year: 2026, Type: Question_Paper

--- Enriched Segmented Questions ---
Question 1 (ID: 4.):
  Type: Definition
  Keywords: 1, home, frame, 3, public
  Text: Indicate your word count at the end of your .
… A FAVOURABLE ALTERNATIVE
What if improving your physical and mental health was as easy as riding in a taxi, bus
or train? Breathing fresh air, doing phy...

--------------------

DataFrame Info after enrichment:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 168 entries, 0 to 167
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   file_path            168 non-null    object
 1   filename             168 non-null    object
 2   raw_text             168 non-null    obje

## Stage 5: Generating the Structured Question Bank

This final stage aggregates all the processed data into a clean, structured question bank. We will achieve this by:

1.  **Exploding the `segmented_questions` column**: Each row in `df_raw_documents` currently represents a document, with the `segmented_questions` column containing a list of individual questions (dictionaries). We need to transform this so that each question becomes its own row in a new DataFrame.
2.  **Structuring the Question Bank**: Create a new DataFrame (`df_question_bank`) where each row is a unique question, including all the metadata from the original document (subject, year, document type, filename) and the question-specific metadata (question ID, text, type, keywords).
3.  **Saving the Question Bank**: Persist the final `df_question_bank` to Google Drive in a suitable format (e.g., Parquet) for easy access and further use.

In [15]:
# Explode the 'segmented_questions' column into new rows
# This creates a new row for each question within each document, duplicating document-level metadata.
print("Exploding segmented questions into a new DataFrame...")
df_question_bank = df_raw_documents.explode('segmented_questions').reset_index(drop=True)

# Extract individual question details into separate columns
# Add a check to ensure 'x' is a dictionary before calling .get()
df_question_bank['question_id'] = df_question_bank['segmented_questions'].apply(lambda x: x.get('question_id') if isinstance(x, dict) else None)
df_question_bank['question_text'] = df_question_bank['segmented_questions'].apply(lambda x: x.get('question_text') if isinstance(x, dict) else None)
df_question_bank['question_type'] = df_question_bank['segmented_questions'].apply(lambda x: x.get('question_type') if isinstance(x, dict) else None)
df_question_bank['keywords'] = df_question_bank['segmented_questions'].apply(lambda x: x.get('keywords') if isinstance(x, dict) else None)

# Drop the original 'segmented_questions' column as its content has been expanded
df_question_bank = df_question_bank.drop(columns=['segmented_questions'])

# Filter out rows where question_id or question_text might be missing (e.g., if a document had no valid questions)
df_question_bank = df_question_bank.dropna(subset=['question_id', 'question_text'])

print("Structured question bank created.")

# Display the first few rows and info of the new DataFrame
print("\nQuestion Bank DataFrame Info:")
df_question_bank.info()
display(df_question_bank.head())

# Display sample question with keywords and type
print("\nSample Question from Question Bank (first row with keywords and type):")
# Ensure there are rows with keywords before trying to access iloc[0]
if not df_question_bank.loc[df_question_bank['keywords'].apply(lambda x: isinstance(x, list) and len(x) > 0)].empty:
    sample_question = df_question_bank.loc[df_question_bank['keywords'].apply(lambda x: isinstance(x, list) and len(x) > 0)].iloc[0]
    print(f"File: {sample_question['filename']}")
    print(f"Subject: {sample_question['subject']}, Year: {sample_question['year']}, Type: {sample_question['document_type']}")
    print(f"Question ID: {sample_question['question_id']}")
    print(f"Question Type: {sample_question['question_type']}")
    print(f"Keywords: {', '.join(sample_question['keywords'])}")
    print(f"Text: {sample_question['question_text'][:500]}...")
else:
    print("No questions with keywords found to display a sample.")

Exploding segmented questions into a new DataFrame...
Structured question bank created.

Question Bank DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
Index: 146 entries, 2 to 264
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   file_path      146 non-null    object
 1   filename       146 non-null    object
 2   raw_text       146 non-null    object
 3   subject        146 non-null    object
 4   year           146 non-null    int64 
 5   document_type  146 non-null    object
 6   cleaned_text   146 non-null    object
 7   question_id    146 non-null    object
 8   question_text  146 non-null    object
 9   question_type  146 non-null    object
 10  keywords       146 non-null    object
dtypes: int64(1), object(10)
memory usage: 13.7+ KB


,file_path,filename,raw_text,subject,year,document_type,cleaned_text,question_id,question_text,question_type,keywords
2,/content/drive/MyDrive/Matric_Past_Papers/Othe...,English-Grade-12-NSC-HL-P1-QP-May-June-2026.pdf,Confidential\nSENIOR CERTIFICATE EXAMINATIONS/...,Other_Subjects,2026,Question_Paper,1 Imagine the vast spectrum of all the culture...,4.,Indicate your word count at the end of your .\...,Definition,"[1, home, frame, 3, public]"
3,/content/drive/MyDrive/Matric_Past_Papers/Othe...,English-Grade-12-NSC-HL-P2-QP-May-June-2026.pdf,Confidential\nSENIOR CERTIFICATE EXAMINATIONS/...,Other_Subjects,2026,Question_Paper,1. Read these instructions carefully before yo...,1.,Read these instructions carefully before you b...,General/Other,"[read, instructions, carefully, begin, answer]"
4,/content/drive/MyDrive/Matric_Past_Papers/Othe...,English-Grade-12-NSC-HL-P2-QP-May-June-2026.pdf,Confidential\nSENIOR CERTIFICATE EXAMINATIONS/...,Other_Subjects,2026,Question_Paper,1. Read these instructions carefully before yo...,2.,Do not attempt to read the entire question pap...,Choice/Selection,"[read, questions, attempt, entire, question]"
5,/content/drive/MyDrive/Matric_Past_Papers/Othe...,English-Grade-12-NSC-HL-P2-QP-May-June-2026.pdf,Confidential\nSENIOR CERTIFICATE EXAMINATIONS/...,Other_Subjects,2026,Question_Paper,1. Read these instructions carefully before yo...,4.,"Answer FIVE questions in all: THREE in , ONE i...",General/Other,"[answer, one, question, questions, five]"
6,/content/drive/MyDrive/Matric_Past_Papers/Othe...,English-Grade-12-NSC-HL-P2-QP-May-June-2026.pdf,Confidential\nSENIOR CERTIFICATE EXAMINATIONS/...,Other_Subjects,2026,Question_Paper,1. Read these instructions carefully before yo...,5.,CHOICE OF ANSWERS FOR SECTIONS B (NOVEL) AND C...,Identification/Listing,"[answer, question, essay, contextual, novel]"



Sample Question from Question Bank (first row with keywords and type):
File: English-Grade-12-NSC-HL-P1-QP-May-June-2026.pdf
Subject: Other_Subjects, Year: 2026, Type: Question_Paper
Question ID: 4.
Question Type: Definition
Keywords: 1, home, frame, 3, public
Text: Indicate your word count at the end of your .
… A FAVOURABLE ALTERNATIVE
What if improving your physical and mental health was as easy as riding in a taxi, bus
or train? Breathing fresh air, doing physical activity and avoiding stress are a few
well-known steps toward healthy living.
Individuals who use public transport get three times more physical activity per day
than those who do not, by walking to a taxi or bus rank and their final destinations.
Getting active helps lower the risk for serio...


In [16]:
# Define the path to save the final question bank
FINAL_QUESTION_BANK_PATH = '/content/drive/MyDrive/structured_question_bank.parquet'

# Save the DataFrame to Google Drive in Parquet format
print(f"Saving final structured question bank to {FINAL_QUESTION_BANK_PATH}...")
df_question_bank.to_parquet(FINAL_QUESTION_BANK_PATH)
print("Question bank saved successfully!")

print(f"\nTotal questions in bank: {len(df_question_bank)}")

Saving final structured question bank to /content/drive/MyDrive/structured_question_bank.parquet...
Question bank saved successfully!

Total questions in bank: 146


# Task
The goal is to develop a Retrieval-Augmented Generation (RAG) system for educational purposes, specifically for South African Matric exam papers. This involves creating a structured question bank from scraped exam papers, enriching questions with metadata, generating embeddings, and implementing a retrieval mechanism.

## Load Structured Question Bank

### Subtask:
Load the `structured_question_bank.parquet` file from Google Drive into a pandas DataFrame.


**Reasoning**:
The first instruction is to load the `structured_question_bank.parquet` file into a pandas DataFrame. This code block will achieve that and then display the head and info of the DataFrame.



In [17]:
import pandas as pd
import os

# Define the path to the structured question bank file
FINAL_QUESTION_BANK_PATH = '/content/drive/MyDrive/structured_question_bank.parquet'

# Check if the file exists before attempting to load
if os.path.exists(FINAL_QUESTION_BANK_PATH):
    # Load the DataFrame
    df_question_bank = pd.read_parquet(FINAL_QUESTION_BANK_PATH)
    print(f"Successfully loaded structured question bank from {FINAL_QUESTION_BANK_PATH}")

    # Display the first 5 rows
    print("\nFirst 5 rows of the structured question bank:")
    display(df_question_bank.head())

    # Display DataFrame info
    print("\nDataFrame Info:")
    df_question_bank.info()
else:
    print(f"Error: The file {FINAL_QUESTION_BANK_PATH} was not found. Please ensure the previous stages ran successfully.")
    df_question_bank = pd.DataFrame() # Initialize an empty DataFrame to avoid errors later

Successfully loaded structured question bank from /content/drive/MyDrive/structured_question_bank.parquet

First 5 rows of the structured question bank:


,file_path,filename,raw_text,subject,year,document_type,cleaned_text,question_id,question_text,question_type,keywords
2,/content/drive/MyDrive/Matric_Past_Papers/Othe...,English-Grade-12-NSC-HL-P1-QP-May-June-2026.pdf,Confidential\nSENIOR CERTIFICATE EXAMINATIONS/...,Other_Subjects,2026,Question_Paper,1 Imagine the vast spectrum of all the culture...,4.,Indicate your word count at the end of your .\...,Definition,"[1, home, frame, 3, public]"
3,/content/drive/MyDrive/Matric_Past_Papers/Othe...,English-Grade-12-NSC-HL-P2-QP-May-June-2026.pdf,Confidential\nSENIOR CERTIFICATE EXAMINATIONS/...,Other_Subjects,2026,Question_Paper,1. Read these instructions carefully before yo...,1.,Read these instructions carefully before you b...,General/Other,"[read, instructions, carefully, begin, answer]"
4,/content/drive/MyDrive/Matric_Past_Papers/Othe...,English-Grade-12-NSC-HL-P2-QP-May-June-2026.pdf,Confidential\nSENIOR CERTIFICATE EXAMINATIONS/...,Other_Subjects,2026,Question_Paper,1. Read these instructions carefully before yo...,2.,Do not attempt to read the entire question pap...,Choice/Selection,"[read, questions, attempt, entire, question]"
5,/content/drive/MyDrive/Matric_Past_Papers/Othe...,English-Grade-12-NSC-HL-P2-QP-May-June-2026.pdf,Confidential\nSENIOR CERTIFICATE EXAMINATIONS/...,Other_Subjects,2026,Question_Paper,1. Read these instructions carefully before yo...,4.,"Answer FIVE questions in all: THREE in , ONE i...",General/Other,"[answer, one, question, questions, five]"
6,/content/drive/MyDrive/Matric_Past_Papers/Othe...,English-Grade-12-NSC-HL-P2-QP-May-June-2026.pdf,Confidential\nSENIOR CERTIFICATE EXAMINATIONS/...,Other_Subjects,2026,Question_Paper,1. Read these instructions carefully before yo...,5.,CHOICE OF ANSWERS FOR SECTIONS B (NOVEL) AND C...,Identification/Listing,"[answer, question, essay, contextual, novel]"



DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
Index: 146 entries, 2 to 264
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   file_path      146 non-null    object
 1   filename       146 non-null    object
 2   raw_text       146 non-null    object
 3   subject        146 non-null    object
 4   year           146 non-null    int64 
 5   document_type  146 non-null    object
 6   cleaned_text   146 non-null    object
 7   question_id    146 non-null    object
 8   question_text  146 non-null    object
 9   question_type  146 non-null    object
 10  keywords       146 non-null    object
dtypes: int64(1), object(10)
memory usage: 13.7+ KB


## Prepare Question Data for Embedding

### Subtask:
Concatenate the 'question_text', 'keywords', and 'question_type' columns to create a single text field for each question. This combined text will be used to generate robust embeddings, providing more context for retrieval.


**Reasoning**:
Concatenate the 'question_text', 'question_type', and 'keywords' columns into a new 'embedding_text' column, handling potential missing values and formatting keywords into a string, then display the head of the DataFrame.



In [18]:
import numpy as np

def create_embedding_text(row):
    question_text = str(row['question_text']).strip() if pd.notna(row['question_text']) else ''
    question_type = str(row['question_type']).strip() if pd.notna(row['question_type']) else ''

    keywords_list = row['keywords']
    if isinstance(keywords_list, list) and len(keywords_list) > 0:
        keywords_str = ', '.join(keywords_list)
    else:
        keywords_str = ''

    # Combine parts, ensuring only non-empty strings are joined with a separator
    parts = [question_text, question_type, keywords_str]
    return ' '.join(filter(None, parts)).strip()

# Apply the function to create the new embedding_text column
print("Creating 'embedding_text' column...")
df_question_bank['embedding_text'] = df_question_bank.apply(create_embedding_text, axis=1)
print("'embedding_text' column created successfully.")

# Display the first few rows with the new column
print("\nFirst 5 rows of df_question_bank with 'embedding_text':")
display(df_question_bank[['question_text', 'question_type', 'keywords', 'embedding_text']].head())

Creating 'embedding_text' column...
'embedding_text' column created successfully.

First 5 rows of df_question_bank with 'embedding_text':


,question_text,question_type,keywords,embedding_text
2,Indicate your word count at the end of your .\...,Definition,"[1, home, frame, 3, public]",Indicate your word count at the end of your .\...
3,Read these instructions carefully before you b...,General/Other,"[read, instructions, carefully, begin, answer]",Read these instructions carefully before you b...
4,Do not attempt to read the entire question pap...,Choice/Selection,"[read, questions, attempt, entire, question]",Do not attempt to read the entire question pap...
5,"Answer FIVE questions in all: THREE in , ONE i...",General/Other,"[answer, one, question, questions, five]","Answer FIVE questions in all: THREE in , ONE i..."
6,CHOICE OF ANSWERS FOR SECTIONS B (NOVEL) AND C...,Identification/Listing,"[answer, question, essay, contextual, novel]",CHOICE OF ANSWERS FOR SECTIONS B (NOVEL) AND C...


## Generate Embeddings for Questions

### Subtask:
Use a pre-trained sentence transformer model to generate vector embeddings for the prepared question texts.


**Reasoning**:
Install the `sentence-transformers` library using `pip`.



In [19]:
!pip install sentence-transformers
print("Sentence-transformers library installed.")

Sentence-transformers library installed.


**Reasoning**:
Now that the `sentence-transformers` library is installed, I will load a pre-trained SentenceTransformer model and use it to generate embeddings for the 'embedding_text' column in `df_question_bank`. This will create a new column named 'question_embedding' to store these vectors.



In [21]:
from sentence_transformers import SentenceTransformer
import torch

# Check for GPU availability and set device
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("Using GPU for embeddings.")
else:
    device = torch.device("cpu")
    print("Using CPU for embeddings.")

# Load a pre-trained sentence transformer model
# 'all-MiniLM-L6-v2' is a good balance of speed and performance for many tasks.
print(f"Loading SentenceTransformer model: 'all-MiniLM-L6-v2' on {device}...")
model = SentenceTransformer('all-MiniLM-L6-v2', device=device)
print("Model loaded successfully.")

# Generate embeddings for the 'embedding_text' column
print("Generating embeddings for questions...")
# Ensure the 'embedding_text' column contains only strings
df_question_bank['embedding_text'] = df_question_bank['embedding_text'].astype(str)
question_embeddings = model.encode(df_question_bank['embedding_text'].tolist(), show_progress_bar=True)

df_question_bank['question_embedding'] = question_embeddings.tolist()
print("Embeddings generated and added to 'question_embedding' column.")

# Display the shape of the embeddings and a sample row with the new column
print(f"Shape of question_embeddings: {question_embeddings.shape}")
print("\nFirst 5 rows of df_question_bank with 'question_embedding':")
display(df_question_bank[['embedding_text', 'question_embedding']].head())

Using CPU for embeddings.
Loading SentenceTransformer model: 'all-MiniLM-L6-v2' on cpu...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded successfully.
Generating embeddings for questions...


Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Embeddings generated and added to 'question_embedding' column.
Shape of question_embeddings: (146, 384)

First 5 rows of df_question_bank with 'question_embedding':


,embedding_text,question_embedding
2,Indicate your word count at the end of your .\...,"[0.0850367620587349, 0.00010662517161108553, 0..."
3,Read these instructions carefully before you b...,"[0.04800505191087723, -0.035214535892009735, -..."
4,Do not attempt to read the entire question pap...,"[0.024708062410354614, 0.04162077233195305, 0...."
5,"Answer FIVE questions in all: THREE in , ONE i...","[0.039703283458948135, 0.01740696094930172, 0...."
6,CHOICE OF ANSWERS FOR SECTIONS B (NOVEL) AND C...,"[0.003098420798778534, 0.030773190781474113, 0..."


What We've Done:
Completed the ETL Pipeline for Question Bank Creation:

Data Ingestion & Setup: We successfully ingested your raw exam paper documents from Google Drive, handling various formats (PDFs with text layers or scanned images) and extracting raw text.
Text Cleaning & Preprocessing: We applied a robust cleaning pipeline to remove noise, boilerplate, and irrelevant text, resulting in clean, normalized text for each document.
Question Segmentation: We developed a specialized parser to segment the cleaned documents into individual questions. Crucially, we ensured that all contextual text, instructions, and passages were kept associated with their respective questions, making the data rich for RAG purposes.
Metadata Enrichment & Tagging: Each segmented question was further enriched by automatically identifying its likely Question Type (e.g., 'Definition', 'Analysis') and extracting relevant Keywords.
Structured Question Bank Generation: Finally, we transformed the processed data into a comprehensive df_question_bank DataFrame, where each row is an individual, enriched question. This DataFrame was then saved to your Google Drive as structured_question_bank.parquet.
Initiated Retrieval-Augmented Generation (RAG) Setup (First Steps):

Loaded Structured Question Bank: We successfully loaded the structured_question_bank.parquet file back into a DataFrame.
Prepared Question Data for Embedding: We created a consolidated embedding_text column for each question, combining its question_text, question_type, and keywords to provide rich context for generating embeddings.
Generated Embeddings for Questions: We installed the sentence-transformers library, loaded a pre-trained model (all-MiniLM-L6-v2), and generated high-dimensional vector embeddings for each question's embedding_text, storing them in a new question_embedding column.
Next Steps for Tomorrow's Implementation (RAG System):
Here’s a plan for how we can proceed to build out the core RAG functionality:

Initialize and Populate Vector Database: We will set up a vector database (such as FAISS for efficient similarity search) and populate it with the generated question_embedding vectors. Each embedding will be indexed and linked to its original question text and all associated metadata (ID, subject, year, document type, etc.).

Implement RAG Retrieval Mechanism: We'll develop a function that takes a user query as input, converts that query into an embedding using the same model, then uses the vector database to find and retrieve the top N most semantically similar questions from your question bank.

Integrate with a Language Model (LLM) (Optional, but often the 'Generation' part of RAG): (This would be a subsequent step if you want the system to generate answers based on retrieved context, rather than just retrieve questions.) We would then feed the retrieved relevant questions (and their context) into an LLM to generate a coherent answer or response to the user's original query.

This structured approach will provide you with a powerful RAG system capable of intelligently finding and utilizing your exam paper questions.